# 7차시 실습 — 유튜브 동영상 검색 시스템

강의(ByteByteGo 「YouTube Video Search」)의 설계를 순서대로 작게 구현합니다.

| 단계 | 강의의 설계 | 이 노트북에서 | 슬라이드 |
|---|---|---|---|
| 0 | ⟨동영상, 질의⟩ 주석 데이터 | 작은 데이터셋 만들기 | 18 |
| 1 | 텍스트 준비: 정규화 → 토큰화 → ID | `normalize`, `TextVectorization` | 20–22 |
| 2 | 동영상 준비: 프레임 디코딩 → 샘플링 → 리사이즈·정규화 | 16프레임 → 4프레임 샘플링 | 28–29 |
| 3 | 두 인코더: 텍스트 인코더 · 프레임 수준 동영상 인코더 | 임베딩 층 / CNN 프레임 임베딩 평균 | 31–42 |
| 4 | 대조학습: 유사도 → softmax → 교차엔트로피 | 배치 안 유사도 행렬로 학습 | 44–46 |
| 5 | 오프라인 평가: MRR | `mrr()` | 49–50 |
| 6 | 서빙: 인덱싱 → 시각 검색 + 텍스트 검색 → 융합 | `search()` | 53–58 |

**학생이 작성할 곳은 세 군데**(4단계 정답 라벨, 5단계 MRR, 6단계 융합)뿐이며 모두 한 줄입니다. `____`(밑줄 4개)를 채우고, 바로 아래 ✅ 셀로 확인하세요.
나머지 셀은 실행하면서 출력과 설명을 읽으면 됩니다. **환경**: Google Colab, 위에서부터 순서대로 실행.

## 0. 준비 — ⟨동영상, 질의⟩ 데이터셋 (슬라이드 18)

강의의 학습 데이터는 **⟨동영상, 텍스트 질의⟩ 쌍**입니다. 같은 모양의 작은 데이터셋을 만듭니다.
- 동영상 = 10가지 대상(개, 고양이, 자동차 …) 중 하나가 찍힌 영상 (2단계에서 CIFAR-10 사진으로 만듦)
- 질의 = 그 영상을 찾으려고 사람이 쓴 문장 → **질의마다 정답 동영상은 1개**
- 제목 = 업로더가 단 제목 → 텍스트 검색(6단계)이 사용. 약 30%는 `IMG_0412` 처럼 의미 없는 제목
- 같은 대상을 여러 단어로 부른다 (dog / puppy / doggo) — 동의어

In [ ]:
import re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers, ops

rng = np.random.default_rng(42)
keras.utils.set_random_seed(42)

CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
SYNONYMS = {"airplane": ["airplane", "plane", "jet"], "automobile": ["car", "automobile", "sedan"],
            "bird": ["bird", "sparrow", "birdie"],     "cat": ["cat", "kitten", "kitty"],
            "deer": ["deer", "fawn", "doe"],           "dog": ["dog", "puppy", "doggo"],
            "frog": ["frog", "toad", "froggy"],        "horse": ["horse", "pony", "stallion"],
            "ship": ["ship", "boat", "vessel"],        "truck": ["truck", "lorry", "pickup"]}
CONTEXTS = ["at night", "in the snow", "on the beach", "in the city",
            "in the rain", "at sunset", "in the park", "on the farm"]

rows = []
for i in range(2000):
    label, ctx = rng.choice(CLASSES), rng.choice(CONTEXTS)
    q_word, t_word = rng.choice(SYNONYMS[label]), rng.choice(SYNONYMS[label])
    query = str(rng.choice([f"{q_word} {ctx}", f"a {q_word} {ctx}", f"video of a {q_word} {ctx}"]))
    title = f"IMG_{i:04d}" if rng.random() < 0.3 else str(rng.choice([f"My {t_word.upper()} {ctx}!!", f"Cute {t_word} {ctx}"]))
    rows.append({"video_id": f"v{i:04d}", "label": str(label), "query": query, "title": title})

df = pd.DataFrame(rows)
df["split"] = ["train"] * 1500 + ["val"] * 200 + ["test"] * 300
df.head(6)

---
## 1. 텍스트 준비 — 정규화 → 토큰화 → 토큰 ID (슬라이드 20–22)

1. **정규화**: 소문자화 · 악센트 제거(NFKD) · 구두점 제거 · 공백 정리 → `'My DOG!!'` 와 `'my dog'` 를 같게
2. **토큰화 + ID 변환**: Keras `TextVectorization` 이 단어로 자르고, 학습 질의로 만든 **룩업 테이블**로 각 단어를 정수 ID로 바꿉니다. 처음 보는 단어는 `[UNK]`(ID 1), 길이를 맞추는 빈칸은 0.

In [ ]:
def normalize(text):
    text = unicodedata.normalize("NFKD", text.lower())                     # 소문자화 + NFKD
    text = "".join(ch for ch in text if not unicodedata.combining(ch))     # 악센트 제거 (é → e)
    text = re.sub(r"[^\w\s]", " ", text)                                   # 구두점 제거
    return re.sub(r"\s+", " ", text).strip()                               # 공백 정리


SEQ_LEN = 8                                                                # 질의 길이를 8 토큰으로 맞춘다
vectorizer = layers.TextVectorization(output_sequence_length=SEQ_LEN, standardize=None)
train = df[df.split == "train"]
vectorizer.adapt([normalize(q) for q in train["query"]])                   # 학습 질의로 룩업 테이블 만들기


def to_ids(texts):
    return vectorizer([normalize(t) for t in texts]).numpy()


print("어휘(룩업 테이블) 크기:", vectorizer.vocabulary_size())
for q in ["Video of a PUPPY at night!!", "a puppy surfing in Montréal"]:
    print(f"{q!r:32s} → {normalize(q)!r:28s} → {to_ids([q])[0]}")

---
## 2. 동영상 준비 — 프레임 → 샘플링 → 리사이즈·정규화 (슬라이드 28–29)

실제 동영상 대신, CIFAR-10 사진 1장을 **카메라가 천천히 움직이듯 잘라 낸 16프레임 영상**을 만듭니다.
동영상 = `(프레임 수, 높이, 너비, 색)` 의 4차원 배열입니다.

> 아래 셀은 사진을 불러오는 부분이라 **실행만** 하면 됩니다. 실습에 필요한 사진만 담은 작은 파일(약 6MB)을 먼저 찾고, 없으면 CIFAR-10 전체(약 170MB)를 한 번 받습니다.

In [ ]:
# (실행만 하세요) 동영상마다 그 대상의 CIFAR-10 사진 1장
import os
SUBSET_FILE, SUBSET_URL = "cifar10_youtube_lab.npz", "https://joy-bae.github.io/cifar10_youtube_lab.npz"
labels = df.label.map(CLASSES.index).values


def load_images():
    for src in [SUBSET_FILE, SUBSET_URL]:                                   # ① 내 폴더 → ② 교수자 GitHub
        try:
            path = src if src == SUBSET_FILE else keras.utils.get_file(SUBSET_FILE, src)
            d = np.load(path)
            if np.array_equal(d["labels"], labels):
                return d["images"]
        except Exception:
            pass
    (x, y), _ = keras.datasets.cifar10.load_data()                          # ③ 원본 CIFAR-10 (처음 한 번만)
    pick = np.random.default_rng(0)
    images = np.stack([x[pick.choice(np.where(y.ravel() == c)[0])] for c in labels])
    np.savez_compressed(SUBSET_FILE, images=images, labels=labels)
    return images


try:
    images = load_images()
except Exception as e:                       # 인터넷이 막힌 환경용 대체 (수업에서는 쓰이지 않음)
    print("⚠️ CIFAR-10 을 받을 수 없어 합성 이미지로 대신합니다:", type(e).__name__)
    color = np.random.default_rng(0).integers(0, 256, (10, 3))
    images = np.clip(color[labels][:, None, None, :] + np.random.default_rng(1).normal(0, 40, (len(labels), 32, 32, 3)),
                     0, 255).astype("uint8")
print("사진:", images.shape)

In [ ]:
def make_video(img, n_frames=16):
    # 사진을 40×40 으로 키운 뒤, 32×32 창을 왼쪽 위 → 오른쪽 아래로 조금씩 옮기며 16장을 잘라 낸다 (패닝)
    big = tf.image.resize(img, (40, 40)).numpy()
    return np.stack([big[t // 2: t // 2 + 32, t // 2: t // 2 + 32] for t in range(n_frames)])


video = make_video(images[0])
print("동영상 1개:", video.shape, "= (프레임 수, 높이, 너비, 색)")
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for t, ax in enumerate(axes.ravel()):
    ax.imshow(video[t].astype("uint8")); ax.axis("off"); ax.set_title(f"t={t}", fontsize=8)
plt.show()

**프레임 샘플링**: 16장을 모두 쓰면 계산량이 커지므로 **균등한 간격으로 4장**만 고릅니다 (0, 5, 10, 15번).
그리고 사전학습 모델이 학습 때 쓴 규칙대로 **리사이즈(96×96) · 정규화(`preprocess_input`)** 합니다.

In [ ]:
from keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input

SAMPLE_IDX = np.linspace(0, 15, 4).astype(int)           # [0, 5, 10, 15]


def preprocess_video(img):
    frames = make_video(img)[SAMPLE_IDX]                  # 16프레임 → 4프레임 샘플링
    frames = tf.image.resize(frames, (96, 96))            # 리사이즈
    return preprocess_input(frames)                       # 정규화 (0~255 → -1~1)


print("샘플링한 프레임 번호:", SAMPLE_IDX, "→ 전처리 후:", preprocess_video(images[0]).shape)

---
## 3. 두 인코더 (슬라이드 31–42)

### 3-1. 동영상 인코더 — 프레임 수준 모델 (슬라이드 41–43)

① 샘플링한 프레임마다 **사전학습 이미지 모델**로 벡터를 만들고 ② **평균**을 내서 동영상 벡터 1개를 만듭니다.
강의의 설계는 ViT 이지만, 실습은 CPU에서 빨리 돌도록 MobileNetV2(5차시와 같은 모델)를 씁니다.
이미지 모델은 이미 학습된 것을 그대로 쓰므로(고정), 모든 동영상의 프레임 평균 벡터를 **미리 한 번만** 계산해 둡니다 (1~2분).

In [ ]:
try:
    frame_model = MobileNetV2(input_shape=(96, 96, 3), include_top=False, pooling="avg", weights="imagenet")
except Exception:                            # 가중치를 받을 수 없는 환경용 (수업에서는 쓰이지 않음)
    frame_model = MobileNetV2(input_shape=(96, 96, 3), include_top=False, pooling="avg", weights=None)

all_frames = np.concatenate([preprocess_video(img) for img in images])     # (동영상 수 × 4, 96, 96, 3)
frame_emb = frame_model.predict(all_frames, batch_size=256, verbose=0)     # 프레임마다 1280차원 벡터
frame_emb = frame_emb.reshape(len(images), len(SAMPLE_IDX), -1)            # (동영상 수, 4, 1280)
video_feat = frame_emb.mean(axis=1)                                        # 프레임 평균 → (동영상 수, 1280)
from sklearn.preprocessing import StandardScaler                           # 표준화: 정형 데이터와 같이 학습 데이터로 fit
video_feat = StandardScaler().fit(video_feat[(df.split == "train").values]).transform(video_feat)
print("동영상 특징:", video_feat.shape)

이 1280차원 평균 벡터 위에 **학습할 층(Dense)** 을 하나 얹으면 동영상 인코더가 완성됩니다.

### 3-2. 텍스트 인코더 (슬라이드 34–36)

토큰 ID → **임베딩 층**(학습되는 룩업 테이블) → 단어 벡터들의 평균 → Dense.
강의의 설계는 BERT 이지만, 실습은 빨리 학습되도록 가장 단순한 'ML 방법'인 임베딩 층을 씁니다.

두 인코더 모두 마지막에 **길이를 1로 맞춥니다**(`UnitNormalization`) → 두 벡터의 내적 = 코사인 유사도 (슬라이드 12).

In [ ]:
EMB_DIM = 64                                                               # 질의와 동영상이 만나는 '같은 공간'의 차원

video_encoder = keras.Sequential([
    keras.Input(shape=(1280,)),
    layers.Dense(EMB_DIM),
    layers.UnitNormalization(),
], name="video_encoder")

text_encoder = keras.Sequential([
    keras.Input(shape=(SEQ_LEN,), dtype="int64"),
    layers.Embedding(vectorizer.vocabulary_size(), 64, mask_zero=True),    # 학습되는 룩업 테이블
    layers.GlobalAveragePooling1D(),                                       # 단어 벡터들의 평균 = 문장 벡터
    layers.Dense(EMB_DIM),
    layers.UnitNormalization(),
], name="text_encoder")

print(video_encoder(video_feat[:2]).shape, text_encoder(to_ids(["a dog at night", "a cat"])).shape)

---
## 4. 모델 훈련 — 대조학습 (슬라이드 44–46)

⟨동영상, 질의⟩ 쌍 B개를 한 번에 넣으면:
1. 동영상 B개, 질의 B개를 각각 인코딩
2. **유사도 행렬** `sim = 동영상 · 질의ᵀ / τ` (B × B) — `sim[i, j]` 는 동영상 i 와 질의 j 의 유사도
3. 각 행은 "동영상 i 의 짝은 B개 질의 중 몇 번인가?" 라는 **B지선다 분류** → softmax + 교차엔트로피
4. 짝은 같은 위치에 있으므로 **정답 = 대각선**: 0번 행의 정답은 0, 1번 행은 1, … → `[0, 1, 2, …, B-1]`

같은 배치의 다른 질의 B−1개가 저절로 음성(negative)이 됩니다. τ(온도)는 확률을 뾰족하게 만드는 값입니다.

**✏️ 작성**: 정답 라벨 한 줄. 힌트: `ops.arange(n)` 은 `[0, 1, …, n-1]`

In [ ]:
TAU = 0.1


def contrastive_loss(_, sim):
    # sim: (B, B) 유사도 행렬. 첫 번째 인자(정답)는 쓰지 않는다 — 정답은 언제나 대각선이므로 여기서 만든다.
    labels = ops.arange(ops.shape(sim)[0])                                      # 정답 = 대각선 위치 [0, 1, …, B-1]
    return keras.losses.sparse_categorical_crossentropy(labels, sim, from_logits=True)

In [ ]:
# ✅ 확인 — 짝끼리 완벽히 같은 방향이면 손실 ≈ 0, 무작위면 손실 ≈ log(B)
perfect = ops.eye(4) / TAU
print("완벽히 맞춘 경우:", round(float(ops.mean(contrastive_loss(None, perfect))), 4))
print("모두 같은 점수   :", round(float(ops.mean(contrastive_loss(None, ops.zeros((4, 4))))), 4), "≈ log 4 =", round(np.log(4), 4))
assert float(ops.mean(contrastive_loss(None, perfect))) < 0.01
print("✅ 통과")

두 인코더를 하나의 학습용 모델로 묶습니다. 입력 = (동영상 특징, 질의 ID), 출력 = 유사도 행렬.
정형 데이터 수업에서 쓰던 `compile → fit` 그대로이며, 검증 손실이 더 내려가지 않으면 멈추는 **EarlyStopping** 을 씁니다.

In [ ]:
v_in = keras.Input(shape=(1280,))
q_in = keras.Input(shape=(SEQ_LEN,), dtype="int64")
sim = layers.Lambda(lambda x: ops.matmul(x[0], ops.transpose(x[1])) / TAU)([video_encoder(v_in), text_encoder(q_in)])
trainer = keras.Model([v_in, q_in], sim)
trainer.compile(optimizer=keras.optimizers.Adam(1e-3), loss=contrastive_loss)

is_tr, is_va = (df.split == "train").values, (df.split == "val").values
X_tr = [video_feat[is_tr], to_ids(df["query"][is_tr])]
X_va = [video_feat[is_va], to_ids(df["query"][is_va])]
history = trainer.fit(X_tr, np.zeros(is_tr.sum()), validation_data=(X_va, np.zeros(is_va.sum())),
                      batch_size=64, epochs=30, verbose=0,
                      callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

plt.figure(figsize=(5, 3))
plt.plot(history.history["loss"], label="train"); plt.plot(history.history["val_loss"], label="val")
plt.axhline(np.log(64), ls=":", c="gray", label="log(64): random guess")
plt.xlabel("epoch"); plt.ylabel("contrastive loss"); plt.legend(); plt.show()

> 손실이 0까지 내려가지 않는 이유: 같은 배치에 같은 대상의 영상이 여러 개 있으면 정답이 아닌데도 비슷할 수밖에 없습니다(거짓 음성). 또 영상 화면에는 `at night` 같은 장소·시간 정보가 없어서, 같은 대상의 영상끼리는 구분할 근거가 없습니다.

---
## 5. 오프라인 평가 — MRR (슬라이드 49–50)

질의마다 정답 동영상은 1개. 테스트 질의로 검색해 **정답이 몇 등에 나왔나(rank)** 를 구하고, `1/rank` 를 평균 낸 것이 MRR 입니다.

**✏️ 작성**: `mrr()` 한 줄.

In [ ]:
def rank_of_answer(S):
    # S[i, j] = 질의 i 와 동영상 j 의 점수. 질의 i 의 정답은 동영상 i (같은 행 번호).
    S = S + rng.uniform(0, 1e-9, S.shape)                     # 동점은 무작위로 순서 정하기
    answer = S[np.arange(len(S)), np.arange(len(S))]          # 대각선 = 정답 동영상의 점수
    return 1 + (S > answer[:, None]).sum(axis=1)              # 정답보다 점수가 높은 동영상 수 + 1 = 순위


def mrr(ranks):
    return np.mean(1 / np.asarray(ranks))                     # 1/순위 의 평균

In [ ]:
# ✅ 확인 — 슬라이드 50 의 예
assert abs(mrr([1, 3, 2, 15, 50]) - 0.384) < 0.001
print("✅ 통과:", round(mrr([1, 3, 2, 15, 50]), 3))

---
## 6. 서빙 — 인덱싱 · 시각 검색 · 텍스트 검색 · 융합 (슬라이드 53–58)

### 6-1. 인덱싱 파이프라인 (미리 해 두는 일, 슬라이드 58)
- **동영상 인덱싱**: 학습된 동영상 인코더로 검색 대상(테스트 동영상 300개)의 임베딩을 미리 계산해 저장
- **텍스트 인덱싱**: 제목을 정규화·토큰화해서 **역색인**(단어 → 동영상 목록)과 TF-IDF 표를 만들어 둠 (Elasticsearch 역할)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

is_te = (df.split == "test").values
db = df[is_te].reset_index(drop=True)                         # 검색 대상 동영상 300개

video_index = video_encoder.predict(video_feat[is_te], verbose=0)            # 동영상 인덱스: (300, 64)

inverted_index = {}                                                         # 역색인: 단어 → 동영상 번호들
for i, title in enumerate(db.title):
    for word in normalize(title).split():
        inverted_index.setdefault(word, set()).add(i)
tfidf = TfidfVectorizer(preprocessor=normalize).fit(db.title)
title_matrix = tfidf.transform(db.title)                                    # 제목의 TF-IDF 표

print("동영상 인덱스:", video_index.shape, "| 역색인 단어 수:", len(inverted_index))
print("'dog' 가 제목에 있는 동영상:", sorted(inverted_index.get("dog", []))[:8], "…")

### 6-2. 예측 파이프라인 (질의가 올 때마다, 슬라이드 54–55)
- **시각 검색**: 질의를 텍스트 인코더로 벡터화 → 동영상 인덱스와 **내적**. (실서비스는 10억 개라 ANN 으로 근사 검색, 여기서는 300개라 전부 계산)
- **텍스트 검색**: 질의 단어와 제목의 TF-IDF 코사인 유사도 (겹치는 단어가 없으면 0 = 역색인 후보가 아님)

In [ ]:
def visual_search(queries):
    q = text_encoder.predict(to_ids(queries), verbose=0)       # 질의 → (질의 수, 64)
    return q @ video_index.T                                   # 내적 → (질의 수, 300)


def text_search(queries):
    return (tfidf.transform(queries) @ title_matrix.T).toarray()


S_visual = visual_search(list(db["query"]))
S_text = text_search(list(db["query"]))
pd.DataFrame({"방법": ["시각 검색만", "텍스트 검색만"],
              "MRR": [mrr(rank_of_answer(S_visual)), mrr(rank_of_answer(S_text))]}).round(3)

### 6-3. 융합 층 (슬라이드 56–57)

두 점수는 범위가 다르므로 질의마다 **0~1 로 맞춘 뒤(min-max)** 가중합합니다.
`최종 점수 = α × 시각 점수 + (1 − α) × 텍스트 점수`

**✏️ 작성**: `fuse()` 한 줄.

In [ ]:
def minmax(S):
    mn, mx = S.min(axis=1, keepdims=True), S.max(axis=1, keepdims=True)
    return (S - mn) / (mx - mn + 1e-9)


def fuse(S_visual, S_text, alpha=0.5):
    return alpha * minmax(S_visual) + (1 - alpha) * minmax(S_text)     # α × minmax(시각) + (1 − α) × minmax(텍스트)

In [ ]:
# ✅ 확인 — 슬라이드 57 의 표: 텍스트 [12, 3, 9], 시각 [0.31, 0.82, 0.60], α = 0.6 → [0.40, 0.60, 0.61]
assert np.allclose(fuse(np.array([[0.31, 0.82, 0.60]]), np.array([[12.0, 3.0, 9.0]]), 0.6).round(2), [[0.40, 0.60, 0.61]])
print("✅ 통과")

for a in [0.0, 0.3, 0.5, 0.7, 1.0]:
    print(f"α = {a:.1f}  →  MRR {mrr(rank_of_answer(fuse(S_visual, S_text, a))):.3f}")
print("(α = 0 은 텍스트만, α = 1 은 시각만. 실무에서는 α 를 검증 데이터로 정한다)")

### 6-4. 완성 — 검색 시스템 써 보기

질의 하나 → 시각 검색 + 텍스트 검색 → 융합 → 상위 5개. 재랭킹(정책 필터 등)은 이 실습에서 생략합니다.

In [ ]:
def search(query, alpha=0.5, k=5):
    scores = fuse(visual_search([query]), text_search([query]), alpha)[0]
    top = np.argsort(-scores)[:k]
    return db.iloc[top][["label", "title"]].assign(score=scores[top].round(3))


search("puppy at night")

**관찰 질문**
1. `search("puppy at night", alpha=0)` (텍스트만)와 `alpha=1` (시각만)의 결과를 비교해 보자. 제목이 `IMG_…` 인 강아지 영상은 어느 쪽에서 찾아지는가? 왜?
2. 시각 검색만 쓰면 `at night` 를 구분하지 못한다. 왜 그럴까? (힌트: 영상 화면에 무엇이 들어 있나)
3. 두 갈래를 융합하면 각각의 단점이 어떻게 보완되는가? (슬라이드 16)

---
## 7. 검색어를 입력받아 동영상 추천하기

지금까지 만든 시스템에 **직접 검색어를 입력**해 봅니다. 결과 표에는 시각 검색 점수와 텍스트 검색 점수, 둘을 융합한 최종 점수가 함께 나옵니다. 추천 동영상의 첫 프레임도 그림으로 보여 줍니다.
- 학습 때 본 적 없는 단어를 넣으면 경고가 나옵니다. 텍스트 인코더가 그 단어를 `[UNK]` 로만 알기 때문입니다 (BERT 같은 사전학습 모델이 필요한 이유).
- `alpha` 를 바꿔 보세요: `recommend("puppy at night", alpha=1.0)` 은 시각 검색만, `alpha=0.0` 은 텍스트 검색만 씁니다.

In [ ]:
# 7. 검색어를 입력받아 동영상 추천하기 — 6단계까지 실행한 뒤 사용
db_images = images[is_te]                              # 검색 대상 동영상 300개의 원본 사진 (썸네일용)
known_words = set(vectorizer.get_vocabulary())         # 텍스트 인코더가 학습 때 본 단어들


def recommend(query, alpha=0.5, k=5):
    s_visual = minmax(visual_search([query]))[0]      # 시각 검색 점수 (0~1)
    s_text = minmax(text_search([query]))[0]          # 텍스트 검색 점수 (0~1)
    scores = fuse(visual_search([query]), text_search([query]), alpha)[0]   # 융합 점수
    top = np.argsort(-scores)[:k]                      # 점수가 높은 순서로 k개

    unknown = [w for w in normalize(query).split() if w not in known_words]
    if unknown:
        print("⚠️ 학습 때 본 적 없는 단어 (시각 검색에서는 [UNK] 로 처리):", unknown)

    result = db.iloc[top][["video_id", "label", "title"]].reset_index(drop=True)
    result.index = result.index + 1                    # 순위를 1부터
    result["시각 점수"] = s_visual[top].round(2)
    result["텍스트 점수"] = s_text[top].round(2)
    result["최종 점수"] = scores[top].round(2)
    display(result)

    fig, axes = plt.subplots(1, k, figsize=(2.2 * k, 2.6))        # 추천 동영상의 첫 프레임 (썸네일)
    for rank, (ax, j) in enumerate(zip(axes, top), start=1):
        ax.imshow(make_video(db_images[j])[0].astype("uint8"))
        ax.set_title(f"{rank}. {db.title[j][:16]}", fontsize=8)
        ax.axis("off")
    plt.show()


print("검색어를 영어로 입력하세요 (예: puppy at night, a boat on the beach, IMG 제목 영상도 찾아집니다)")
print("아무것도 입력하지 않고 Enter 를 누르면 끝납니다.")
while True:
    query = input("\n🔍 검색어: ").strip()
    if not query:
        print("검색을 마칩니다.")
        break
    recommend(query)